# Dataset

In [ ]:
!pip install tsai

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta


pd.set_option('display.max_columns', None) # Mostrar todas las columnas al imprimir df

In [11]:
# Lectura de TIP.csv
df_tip = pd.read_csv('Datos/TIP.csv')
df_tip

,NORAD_CAT_ID,MSG_EPOCH,INSERT_EPOCH,DECAY_EPOCH,WINDOW,REV,DIRECTION,LAT,LON,INCL,NEXT_REPORT,ID,HIGH_INTEREST,OBJECT_NUMBER
0,25667,11/08/2025 5:02,11/08/2025 5:16,14/08/2025 19:48,1080,36936,ascending,1.7,40.2,4.3,72,55689,Y,25667
1,25667,12/08/2025 21:45,15/08/2025 22:41,14/08/2025 19:31,840,36934,ascending,-3.2,333.2,4.3,24,55709,Y,25667
2,25667,13/08/2025 20:55,15/08/2025 22:41,13/08/2025 16:24,1,36916,descending,-1.7,281.0,4.3,0,55719,Y,25667
3,63158,4/08/2025 20:33,4/08/2025 20:46,8/08/2025 20:30,1140,488,ascending,16.0,97.2,19.3,72,55611,N,63158
4,63158,8/08/2025 8:13,8/08/2025 8:26,8/08/2025 10:27,240,480,ascending,15.5,247.6,19.2,0,55624,N,63158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,42994,3/01/2025 19:02,3/01/2025 19:18,4/01/2025 1:26,120,39972,descending,-17.3,191.9,97.4,2,37514,N,42994
322,42994,3/01/2025 23:05,3/01/2025 23:18,4/01/2025 1:08,37,39972,descending,53.4,208.6,97.4,0,37521,N,42994
323,42994,4/01/2025 2:44,4/01/2025 3:28,4/01/2025 1:04,1,39972,descending,55.7,209.7,97.4,0,37525,N,42994
324,57804,3/01/2025 7:24,3/01/2025 7:38,3/01/2025 13:28,240,341,ascending,-1.1,15.2,29.8,2,37504,N,57804


In [12]:
# Lectura de TLE.csv
df_tle = pd.read_csv('Datos/TLE.csv')
print(f"Columnas: {df_tle.columns}")
df_tle.head(3)

Columnas: Index(['COMMENT', 'ORIGINATOR', 'NORAD_CAT_ID', 'OBJECT_NAME', 'OBJECT_TYPE',
       'CLASSIFICATION_TYPE', 'INTLDES', 'EPOCH', 'EPOCH_MICROSECONDS',
       'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE',
       'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'ELEMENT_SET_NO',
       'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'FILE',
       'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'OBJECT_ID', 'OBJECT_NUMBER',
       'SEMIMAJOR_AXIS', 'PERIOD', 'APOGEE', 'PERIGEE', 'DECAYED'],
      dtype='object')


,COMMENT,ORIGINATOR,NORAD_CAT_ID,OBJECT_NAME,OBJECT_TYPE,CLASSIFICATION_TYPE,INTLDES,EPOCH,EPOCH_MICROSECONDS,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,FILE,TLE_LINE0,TLE_LINE1,TLE_LINE2,OBJECT_ID,OBJECT_NUMBER,SEMIMAJOR_AXIS,PERIOD,APOGEE,PERIGEE,DECAYED
0,GENERATED VIA SPACETRACK.ORG API,18 SPCS,25667,ARIANE 42P R/B,ROCKET BODY,U,99016B,2025-08-07 12:00:00,0,15.114824,0.055274,4.328200,335.538500,94.409400,222.89200,0,999,36776,0.000817,0.025283,-0.000028,4806722,0 ARIANE 42P R/B,1 25667U 99016B 25219.50000000 .02528315 -2...,2 25667 4.3282 335.5385 0552739 94.4094 222...,1999-016B,25667,6909.815,95.271,913.613,149.748,1
1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,25667,ARIANE 42P R/B,ROCKET BODY,U,99016B,2025-08-07 12:00:00,0,15.114824,0.055274,4.328200,335.538500,94.409400,222.89200,0,999,36820,0.000817,0.025283,-0.000028,4806670,0 ARIANE 42P R/B,1 25667U 99016B 25219.50000000 .02528315 -2...,2 25667 4.3282 335.5385 0552739 94.4094 222...,1999-016B,25667,6909.815,95.271,913.613,149.748,1
2,GENERATED VIA SPACETRACK.ORG API,18 SPCS,25667,ARIANE 42P R/B,ROCKET BODY,U,99016B,2025-08-03 20:04:11,844480,14.773368,0.069171,4.330284,2.619863,40.630842,324.34104,0,999,36775,0.001352,0.031880,-0.000026,4806584,0 ARIANE 42P R/B,1 25667U 99016B 25215.83624820 .03188028 -2...,2 25667 4.3303 2.6199 0691709 40.6308 324...,1999-016B,25667,7015.880,97.473,1123.040,152.451,1


## Clasificadora de órbitas

In [13]:
def get_orbit_type(apogee, perigee):
    """
        Clasifica la órbita en base a su Apogeo y Perigeo (en km).
        - Input: Apogeo y Perigeo en km.
        - Output: Tipo de órbita y clase numérica.
    """
    if apogee < 2000:
        return 'LEO', 0  
    elif perigee < 2000 and apogee > 2000:
        return 'GTO', 1  # En realidad aplica para HEO y GTO, pero lo etiquetamos como GTO para simplificar
    elif perigee >= 2000 and apogee <= 35000:
        return 'MEO', 2
    elif perigee >= 35000:
        return 'GEO', 3
    else:
        return 'UNKNOWN', 4
    
# Fuente principal (ESA):
# - https://conference.sdo.esoc.esa.int/proceedings/sdc5/paper/15

# Otras:
# - https://www.esa.int/Enabling_Support/Space_Transportation/Types_of_orbits
# - https://eos.com/blog/types-of-satellites/

## Preparación ventanas

In [14]:
def deduplicate_tle_rows(tle_df):
    """
    Elimina filas duplicadas de TLE para el mismo objeto (NORAD_CAT_ID) y momento exacto (EPOCH).
    Se queda siempre con la versión que tiene el número de FILE más alto.
    """
    total_inicial = len(tle_df)

    if not pd.api.types.is_datetime64_any_dtype(tle_df['EPOCH']):
        tle_df['EPOCH'] = pd.to_datetime(tle_df['EPOCH'])
        
    tle_df_sorted = tle_df.sort_values(['NORAD_CAT_ID', 'EPOCH', 'FILE'])     # Ordenamos por objeto, fecha y FILE (de menor a mayor)
    tle_df_clean = tle_df_sorted.drop_duplicates(subset=['NORAD_CAT_ID', 'EPOCH'], keep='last')     # Eliminamos duplicados

    tle_df_clean = tle_df_clean.reset_index(drop=True)
    
    eliminadas = total_inicial - len(tle_df_clean)
    
    print(f"[DEBUG] TLEs duplicados eliminados: {eliminadas} (de {total_inicial} filas originales)")
    print(f"[DEBUG] Total de TLEs únicos finales: {len(tle_df_clean)}")
    
    return tle_df_clean

In [15]:
WINDOW_SIZE = 7
DAYS_BEFORE_DECAY_SEARCH = 14

def get_windows(tle, tip, limpiar_cols=True):
    print("PASO 1: Extrayendo y limpiando ventanas de 1 semana exacta...")

    # Formateo de fechas
    tip['MSG_EPOCH'] = pd.to_datetime(tip['MSG_EPOCH'], dayfirst=True)
    tip['DECAY_EPOCH'] = pd.to_datetime(tip['DECAY_EPOCH'], dayfirst=True)
    tle['EPOCH'] = pd.to_datetime(tle['EPOCH'])

    tip = tip.sort_values(['NORAD_CAT_ID', 'MSG_EPOCH']) # Ordenamos por objeto y fecha
    last_tips = tip.groupby('NORAD_CAT_ID').last().reset_index() # Nos quedamos con el último TIP de cada objeto (decay más exacto)

    windows = []
    seq_id = 1

    for index, row in last_tips.iterrows(): # Iteramos por cada objeto usando su último TIP
        obj_id = row['NORAD_CAT_ID'] # Seleccionamos el ID del objeto
        ground_truth_decay = row['DECAY_EPOCH']
        
        obj_tles = tle[tle['NORAD_CAT_ID'] == obj_id].sort_values('EPOCH') # Filtramos TLEs de este objeto y los ordenamos por fecha]

        if obj_tles.empty: continue
        
        primer_tle = obj_tles.iloc[0]
        _, orbit_code = get_orbit_type(primer_tle['APOGEE'], primer_tle['PERIGEE'])
        
        # Inicio y fin de búsqueda para cabezas de ventana (14 a 1 días antes del decay)
        inicio_busqueda = ground_truth_decay - timedelta(days=DAYS_BEFORE_DECAY_SEARCH)
        fin_busqueda = ground_truth_decay - timedelta(days=1)
        
        anclas = obj_tles[(obj_tles['EPOCH'] >= inicio_busqueda) & (obj_tles['EPOCH'] <= fin_busqueda)]
        
        for _, anchor_tle in anclas.iterrows():
            anchor_epoch = anchor_tle['EPOCH']
            
            # --- VENTANA DE TIEMPO DEFINIDA (1 SEMANA) ---
            start_window = anchor_epoch - timedelta(days=WINDOW_SIZE)
            window_tles = obj_tles[(obj_tles['EPOCH'] >= start_window) & (obj_tles['EPOCH'] <= anchor_epoch)].copy()
            
            if window_tles.empty: continue
            
            y_target_minutes = (ground_truth_decay - anchor_epoch).total_seconds() / 60.0
            
            window_tles['SEQUENCE_ID'] = seq_id # ID global de ventana
            window_tles['LOG_BSTAR'] = np.log10(window_tles['BSTAR'].abs().clip(lower=1e-9)) # Transformación logarítmica de B* para mejorar la distribución

            # Etiqueta continua (minutos hasta el decay)
            window_tles['TARGET_Y_MINUTES'] = y_target_minutes 

            # Etiquetas discretas (clases)
            window_tles['TARGET_DAY_CLASS'] = int(y_target_minutes // (24 * 60))
            window_tles['TARGET_HOUR_CLASS'] = int((y_target_minutes % (24 * 60)) // 60)
            window_tles['TARGET_MINUTE_CLASS'] = int(y_target_minutes % 60)
            
            window_tles['ANCHOR_EPOCH'] = anchor_epoch
            window_tles['IS_PADDING'] = 0 
            # window_tles['ORBIT_TYPE'] = orbit_name
            window_tles['ORBIT_CLASS'] = orbit_code
            
            if limpiar_cols:
                window_tles = column_selection(window_tles)
            
            windows.append(window_tles)
            seq_id += 1
            
    return windows

In [16]:
def column_selection(window_tles):
    """
        Función para seleccionar columnas específicas de un DataFrame.
        - window_tles: DataFrame con los TLEs.
        - Retorna un nuevo DF con solo las columnas seleccionadas.
    """
    columns = [
                'SEQUENCE_ID', 'NORAD_CAT_ID', 'EPOCH', 'ANCHOR_EPOCH', 'IS_PADDING',
                # 'ORBIT_TYPE',
                'ORBIT_CLASS',
                'MEAN_MOTION', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'PERIGEE', 'SEMIMAJOR_AXIS',
                'BSTAR', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY',
                'TARGET_Y_MINUTES', 'TARGET_DAY_CLASS', 'TARGET_HOUR_CLASS', 'TARGET_MINUTE_CLASS'
            ]
    return window_tles[columns].copy()

In [17]:
def fill_with_padding(windows):
    if not windows:
        print("No se encontraron datos.")
        return None
            
    # Encontrar el tamaño de la ventana con más TLEs
    max_seq_length = max(len(v) for v in windows)
    print(f"La ventana más grande tiene {max_seq_length} TLEs. Todas las ventanas se rellenarán con padding hasta este tamaño.")
    
    windows_processed = []
    
    # Rellenar cada ventana
    for window_tles in windows:
        faltantes = max_seq_length - len(window_tles)
        
        if faltantes > 0:
            pad_df = pd.DataFrame(index=range(faltantes), columns=window_tles.columns)
            
            # Metadatos que deben conservarse en las filas de relleno
            pad_df['SEQUENCE_ID'] = window_tles.iloc[0]['SEQUENCE_ID']
            pad_df['NORAD_CAT_ID'] = window_tles.iloc[0]['NORAD_CAT_ID']
            pad_df['ANCHOR_EPOCH'] = window_tles.iloc[0]['ANCHOR_EPOCH']
            # pad_df['ORBIT_TYPE'] = window_tles.iloc[0]['ORBIT_TYPE']
            pad_df['ORBIT_CLASS'] = window_tles.iloc[0]['ORBIT_CLASS']
            pad_df['TARGET_Y_MINUTES'] = window_tles.iloc[0]['TARGET_Y_MINUTES']
            pad_df['TARGET_DAY_CLASS'] = window_tles.iloc[0]['TARGET_DAY_CLASS']
            pad_df['TARGET_HOUR_CLASS'] = window_tles.iloc[0]['TARGET_HOUR_CLASS']
            pad_df['TARGET_MINUTE_CLASS'] = window_tles.iloc[0]['TARGET_MINUTE_CLASS']
            pad_df['IS_PADDING'] = 1 # Indicador de que esta fila es relleno
            
            # El resto de columnas (física) se rellenan con 0
            pad_df = pad_df.fillna(0)
            
            # Pre-padding (Ceros arriba, datos reales abajo)
            window_tles = pd.concat([pad_df, window_tles], ignore_index=True) # Concatenamos el df de padding con el df original, colocando el padding antes de los datos reales
            
        windows_processed.append(window_tles)
        
    # Juntar todo el dataset
    df_final = pd.concat(windows_processed, ignore_index=True)
    return df_final, max_seq_length

In [18]:
# --- EJECUCIÓN DEL PIPELINE (MAIN) ---
def preprocessing(limpiar_cols=True):
    ruta_tip='Datos/TIP.csv'
    ruta_tle='Datos/TLE.csv'

    tip = pd.read_csv(ruta_tip)
    tle = pd.read_csv(ruta_tle)
    tle = deduplicate_tle_rows(tle)

    windows = get_windows(tle=tle, tip=tip, limpiar_cols=limpiar_cols)
    
    df_final, max_length = fill_with_padding(windows)
    
    if df_final is not None:
        # Exportar a CSV
        ruta_salida = 'datasets/Dataset_TFM_Limpio.csv'
        df_final.to_csv(ruta_salida, index=False, sep=';', decimal=',')
        
        print("\n¡Pipeline finalizado con éxito!")
        print(f"Total de secuencias de entrenamiento (ventanas): {len(windows)}")
        print(f"Tamaño fijo de la ventana: {max_length}")
        print(f"Filas totales en el CSV: {len(df_final)} ({len(windows)} * {max_length})")
        print(f"Guardado en: {ruta_salida}")

# Ejecutar todo
preprocessing()

[DEBUG] TLEs duplicados eliminados: 650 (de 2731 filas originales)
[DEBUG] Total de TLEs únicos finales: 2081
PASO 1: Extrayendo y limpiando ventanas de 1 semana exacta...
La ventana más grande tiene 34 TLEs. Todas las ventanas se rellenarán con padding hasta este tamaño.


C:\Users\acost\AppData\Local\Temp\ipykernel_24040\1921955322.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pad_df = pad_df.fillna(0)
C:\Users\acost\AppData\Local\Temp\ipykernel_24040\1921955322.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pad_df = pad_df.fillna(0)
C:\Users\acost\AppData\Local\Temp\ipykernel_24040\1921955322.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `p


¡Pipeline finalizado con éxito!
Total de secuencias de entrenamiento (ventanas): 815
Tamaño fijo de la ventana: 34
Filas totales en el CSV: 27710 (815 * 34)
Guardado en: datasets/Dataset_TFM_Limpio.csv


In [ ]:
def pipeline():
    preprocessing()

# 

In [ ]:
user = np.array([1,2]).repeat(4).reshape(-1,1)
val = np.random.rand(8, 3)
data = np.concatenate([user, val], axis=-1)
df = pd.DataFrame(data, columns=['user', 'x1', 'x2', 'x3'])
test_eq(df2np3d(df, ['user'], ['x1', 'x2', 'x3']).shape, (2, 3, 4))